# Local Evidence Classification

Evaluate token-window evidence without leaking parent texts across partitions. MG is the positive class (`1`).

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import display
import pandas as pd

from classification_utils import plot_score_curves
from local_evidence_utils import (
    LOCAL_FEATURE_COLUMNS,
    build_local_evidence_windows,
    default_local_evidence_path,
    load_or_build_local_evidence_windows,
    run_local_evidence_experiments,
    split_local_evidence_texts,
)
from token_candidate_similarity_heatmaps_utils import visualize_text_similarities

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
DATASET = "kaggle_train"
MODEL = "opt_350m"
# WINDOW_LENGTHS = (32, 64, 96, 128, 192, 256)
WINDOW_LENGTHS = (32) # short list for tests

STRIDE = 16
TOP_K = 20
SEED = 42
RECOMPUTE_WINDOWS = False

pd.set_option("display.max_columns", 100)

In [ ]:
WINDOWS_PATH = default_local_evidence_path(PROCESSED_ROOT, DATASET, MODEL)
windows = load_or_build_local_evidence_windows(
        PROCESSED_ROOT,
        DATASET,
        MODEL,
        path=WINDOWS_PATH,
        window_lengths=WINDOW_LENGTHS,
        stride=STRIDE,
        top_k=TOP_K,
        recompute=RECOMPUTE_WINDOWS,
    )

display(windows.groupby(["split", "window_length"]).size().rename("window_rows").reset_index())
display(
    windows.loc[:, LOCAL_FEATURE_COLUMNS]
    .notna()
    .sum()
    .rename("non_null_windows")
    .rename_axis("feature")
    .reset_index()
)

In [ ]:
partitions = split_local_evidence_texts(windows, seed=SEED)
test_summary, test_predictions, scored_windows = run_local_evidence_experiments(
    windows,
    partitions,
    feature_columns=LOCAL_FEATURE_COLUMNS,
    seed=SEED,
)
display(test_summary.sort_values(["window_length", "aggregation"]).reset_index(drop=True))
curve_predictions = test_predictions.assign(
    mg_score=lambda frame: frame["mg_score"].where(
        frame["score_direction"].eq("higher_is_mg"),
        -frame["mg_score"],
    ),
        experiment=lambda frame: frame["window_length"].astype(str) + " tokens: " + frame["aggregation"]
)
plot_score_curves(curve_predictions)

In [ ]:
selected_predictions = test_predictions.assign(
    decision_score=lambda frame: frame["mg_score"].where(
        frame["score_direction"].eq("higher_is_mg"),
        -frame["mg_score"],
    ),
).sort_values("decision_score", ascending=False).reset_index(drop=True)
display(selected_predictions)

selected_evidence = selected_predictions.loc[
    :,
    [
        "split",
        "text_id",
        "label",
        "window_length",
        "prediction",
        "score_direction",
        "mg_score",
        "decision_score",
        "selected_start_token",
        "selected_stop_token",
    ],
].merge(
    scored_windows.loc[
        scored_windows["partition"].eq("test"),
        [
            "split",
            "text_id",
            "window_length",
            "start_token",
            "stop_token",
            "window_text",
            "top_positive_feature",
            "top_negative_feature",
        ],
    ],
    left_on=["split", "text_id", "window_length", "selected_start_token", "selected_stop_token"],
    right_on=["split", "text_id", "window_length", "start_token", "stop_token"],
    how="left",
    validate="many_to_one",
)
display(
    selected_evidence[
        [
            "text_id",
            "prediction",
            "label",
            "score_direction",
            "mg_score",
            "decision_score",
            "selected_start_token",
            "selected_stop_token",
            "window_text",
            "top_positive_feature",
            "top_negative_feature",
        ]
    ]
)

In [ ]:
INSPECT_TEXT = None
INSPECT_SPLIT = "MG"
INSPECT_START_TOKEN = 0
INSPECT_STOP_TOKEN = 128

if INSPECT_TEXT is not None:
    visualize_text_similarities(
        PROCESSED_ROOT,
        DATASET,
        MODEL,
        INSPECT_SPLIT,
        INSPECT_TEXT,
        top_k=TOP_K,
        token_range=(INSPECT_START_TOKEN, INSPECT_STOP_TOKEN),
    )